# Step 5. Paganin phase retrieval and initial tomographic reconstruction

Converts the flat-field-corrected intensity projections into **phase projections** using the
multi-distance Paganin algorithm, then performs a quick **filtered back-projection** (FBP)
tomographic reconstruction to produce the initial guess for the full iterative reconstruction
in Step 6.

**Outputs written to `/exchange/`**
| Dataset | Description |
|---------|-------------|
| `obj_init_re{paganin}_{bin}` | Real part of the complex initial object (3-D volume) |
| `obj_init_imag{paganin}_{bin}` | Imaginary part |

> `paganin` is the δ/β ratio assumed for the sample material.  
> `bin` selects which pyramid level from Step 4 to use.

In [ ]:
import numpy as np
import cupy as cp
import h5py
from holotomocupy.utils import *
from holotomocupy.tomo import Tomo
from holotomocupy.chunking import Chunking
from holotomocupy.shift import Shift


### Initialisation

Read geometry and metadata from the HDF5 file.  Compute wavelength, effective propagation
distances (accounting for magnification), and the object-space voxel size at the chosen bin level.

In [ ]:
ntheta = 1800
st = 0
bin = 0
show = True
rotation_center_shift = -27.00227

paganin = 120
filter_name = 'shepp'  # FBP filter: 'ramp' | 'shepp' | 'parzen'

ids = np.arange(0, 1800, 1800 / ntheta).astype('int')

path_out = '/data2/vnikitin/atomium_rec/20240924/AtomiumS2/'
file_out = f'data.h5'


with h5py.File(f'{path_out}/{file_out}', 'r') as fid:
    detector_pixelsize = fid['/exchange/detector_pixelsize'][0]    
    focusToDetectorDistance = fid['/exchange/focusdetectordistance'][0]    
    z1 = fid['/exchange/z1'][:] 
    energy = fid['/exchange/energy'][0] 
    shifts = fid['/exchange/shifts'][ids]
    theta = fid['/exchange/theta'][ids, 0]
    theta = -theta / 180 * np.pi
    attrs = fid['/exchange/attrs'][ids]
    shape = np.array(fid[f'/exchange/data0'].shape)
    shape_ref = fid['/exchange/data_white_start0'].shape
    shape_dark = fid['/exchange/data_dark0'].shape    

wavelength = 1.24e-09 / energy
z2 = focusToDetectorDistance - z1
magnifications = focusToDetectorDistance / z1
norm_magnifications = magnifications / magnifications[0]
distances = (z1 * z2) / focusToDetectorDistance * norm_magnifications**2
voxelsize = detector_pixelsize / magnifications[0]
ndist = len(z1)
n = shape[1] 
nobj = int(np.ceil((n) / norm_magnifications[-1] / 64)) * 64 
# nobj+=nobj//8
## change to the current bin level
n //= (2**bin)
nobj //= (2**bin)
voxelsize = voxelsize*2**bin

print(f'{energy=}')
print(f'{z1=}')
print(f'{focusToDetectorDistance=}')
print(f'{detector_pixelsize=}')
print(f'{magnifications=}')
print(f'{voxelsize=}')
print(f'{distances=}')

### Read data and apply flat-field correction

Load the preprocessed projections and references from the bin level chosen above.  
Apply the combined shifts (Step 3) scaled to the current bin level, including the
`rotation_center_shift`.

In [ ]:
data = np.zeros([ntheta, ndist, n, n], dtype='float32')
with h5py.File(f'{path_out}/{file_out}', 'r') as fid:
    for k in range(ndist):
        data[:, k] = fid[f'/exchange/pdata{k}_{bin}'][ids]
    ref = fid[f'/exchange/pref_{bin}'][:ndist]
    r = (fid[f'/exchange/cshifts_final'][ids] / 2**bin).astype('float32')
    r[...,0]+=400/2**bin
    #compensate for rotation center shift
    s = rotation_center_shift
    for k in range(bin):
        s = (s - 0.5) / 2
    r[..., 1] += s

### Align and stitch distances into the object frame

Same stitching procedure as Step 4 but applied at the current bin level to produce
`srdata_out` of shape `[ntheta, ndist, nobj, nobj]`.

In [ ]:
cl_shift = Shift(n, nobj, n, nobj, 1 / norm_magnifications, 'complex64')
distances_pag = distances / norm_magnifications**2
npad = n // 16
cref  = cp.array(ref)
r_gpu = cp.array(r)   # move all shifts to GPU once

# Precompute quintic smooth-step (does not depend on j or k)
v = cp.linspace(0, 1, npad, endpoint=False)
v = v**5 * (126 - 420*v + 540*v**2 - 315*v**3 + 70*v**4)

chunk_size = 16
read_buf = [np.empty([chunk_size, n, n], dtype='float32') for _ in range(ndist)]

srdata_out = np.empty([ntheta, ndist, nobj, nobj], dtype='float32')

with h5py.File(f'{path_out}/data.h5', 'a') as fid:
    srdata = cp.zeros([ndist, nobj, nobj], dtype='float32')

    for j0 in range(0, ntheta, chunk_size):
        j1 = min(j0 + chunk_size, ntheta)
        nc = j1 - j0

        # One HDF5 call per distance fills the read buffer for the whole chunk
        for k in range(ndist):
            read_buf[k][:nc] = np.array(fid[f'/exchange/pdata{k}_{bin}'][ids[j0:j1]], dtype='float32')

        for i in range(nc):
            j = j0 + i
            data = cp.empty([ndist, n, n], dtype='float32')
            for k in range(ndist):
                data[k] = cp.array(read_buf[k][i])

            rdata = data / (cref + 1e-5)

            for k in range(ndist - 1, -1, -1):
                tmp = rdata[k].astype('complex64')
                # r_gpu[j:j+1, k] is already on GPU — no H2D transfer needed
                tmp = cl_shift.curlySback(cp.log(tmp[None]).astype('complex64'), r_gpu[j:j+1, k], k)[0].real
                tmp /= norm_magnifications[k]**2
                tmp = cp.exp(tmp)

                padx0 = int((nobj - n / norm_magnifications[k]) / 2) - int(r[j, k, 1])
                pady0 = int((nobj - n / norm_magnifications[k]) / 2) - int(r[j, k, 0])
                padx1 = int((nobj - n / norm_magnifications[k]) / 2) + int(r[j, k, 1])
                pady1 = int((nobj - n / norm_magnifications[k]) / 2) + int(r[j, k, 0])
                padx0 = min(nobj, max(0, padx0)) + 5
                pady0 = min(nobj, max(0, pady0)) + 5
                padx1 = min(nobj, max(0, padx1)) + 5
                pady1 = min(nobj, max(0, pady1)) + 5

                tmp = cp.pad(tmp[pady0:-pady1], ((pady0, pady1), (0, 0)), 'edge')
                tmp = cp.pad(tmp[:, padx0:-padx1], ((0, 0), (padx0, padx1)),
                             'linear_ramp', end_values=((1, 1), (1, 1)))

                if k < ndist - 1:
                    # Intensity match on GPU — float() does a single scalar D2H
                    mmm = float(srdata[k+1][pady0:-pady1, padx0:-padx1].mean() /
                                tmp[pady0:-pady1, padx0:-padx1].mean())
                    tmp *= mmm

                    wx = cp.ones(nobj, dtype='float32')
                    wy = cp.ones(nobj, dtype='float32')
                    wx[:padx0] = 0;  wx[padx0:padx0+npad] = v;  wx[-padx1-npad:-padx1] = 1-v;  wx[-padx1:] = 0
                    wy[:pady0] = 0;  wy[pady0:pady0+npad] = v;  wy[-pady1-npad:-pady1] = 1-v;  wy[-pady1:] = 0

                    w   = cp.outer(wy, wx)
                    tmp = tmp * w + srdata[k+1] * (1 - w)
                srdata[k] = tmp

            srdata_out[j] = srdata.get()
            if j % 100 == 0:
                print(j)
                mshow_complex(srdata[0] + 1j*srdata[ndist-1], show)

srdata = srdata_out

### Multi-distance Paganin phase retrieval

`multiPaganin` implements the frequency-domain Paganin formula for **multiple propagation
distances** simultaneously:

$$
\phi = \frac{\delta}{2} \log \left(
  \mathcal{F}^{-1} \left[
    \frac{ \sum_j (1 + \lambda z_j \pi \frac{\delta}{\beta} |\xi|^2)\, \hat{I}_j(\xi) }
         { \sum_j (1 + \lambda z_j \pi \frac{\delta}{\beta} |\xi|^2)^2 + \alpha }
  \right]
\right)
$$

where $\xi = (f_x, f_y)$ are spatial frequencies, $\lambda$ is the X-ray wavelength,
$z_j$ are the effective propagation distances, and $\delta/\beta$ (`delta_beta`) is the
refractive index ratio (controlled by the `paganin` parameter).

`rec_init` wraps `multiPaganin` for all projections, pads data to reduce wrap-around, subtracts
a background offset, and builds a complex object with the imaginary part estimated as
`real / paganin` (thin-object approximation).

In [ ]:
voxelsize

In [ ]:
def multiPaganin(data, distances, wavelength, voxelsize, delta_beta, alpha):
    fx = cp.fft.fftfreq(data.shape[-1], d=voxelsize).astype('float32')
    [fx, fy] = cp.meshgrid(fx, fx)
    numerator = 0
    denominator = 0
    for j in range(data.shape[0]):        
        rad_freq = cp.fft.fft2(data[j])
        taylorExp = 1 + wavelength * distances[j] * cp.pi * delta_beta * (fx**2 + fy**2)
        numerator += taylorExp * rad_freq
        denominator += taylorExp**2

    numerator /= len(distances)
    denominator = (denominator / len(distances)) + alpha

    phase = cp.log(cp.real(cp.fft.ifft2(numerator / denominator)))
    phase *= delta_beta * 0.5

    return phase

def rec_init(rdata, paganin):
    recPaganin = np.zeros([ntheta, nobj, nobj], dtype='float32')
    mm_fixed = None
    global_bg = None
    for j in range(ntheta):
        pj = cp.array(rdata[j])
        if j == 0:
            mm_fixed  = float(pj[:, :32 * n // 512, :32 * n // 512].mean())
            pj_pad    = cp.pad(pj, ((0, 0), (nobj // 8, nobj // 8), (nobj // 8, nobj // 8)),
                               'constant', constant_values=mm_fixed)
            ph0       = multiPaganin(pj_pad, distances, wavelength, voxelsize, paganin, 1e-5)
            ph0_crop  = ph0[nobj // 8:-nobj // 8, nobj // 8:-nobj // 8]
            global_bg = float(cp.median(ph0_crop[:16 * n // 512, :16 * n // 512]))
        pj    = cp.pad(pj, ((0, 0), (nobj // 8, nobj // 8), (nobj // 8, nobj // 8)),
                       'constant', constant_values=mm_fixed)
        phase = multiPaganin(pj, distances, wavelength, voxelsize, paganin, 1e-5)
        recPaganin[j] = phase[nobj // 8:-nobj // 8, nobj // 8:-nobj // 8].get()

    recPaganin -= global_bg
    psi = np.empty([ntheta, nobj, nobj], dtype='complex64')
    psi.real[:] = recPaganin
    psi.imag[:] = recPaganin / paganin
    return psi

psi_data = rec_init(srdata, paganin)
mshow_complex(psi_data[-1], show)

### Filtered back-projection via `Tomo.fbp` + `Chunking.gpu_batch`

`psi_data` has shape `[ntheta, nobj, nobj]` (complex64, CPU).  
`Tomo.fbp` applies a 1-D frequency-domain filter along the detector axis and then calls `RT`
(adjoint Radon transform) — the classic FBP pipeline in one call:

```
rec = RT( h(ω) * fft(psi_data) )
```

Available filters (`filter_name`):

| Name | Formula | Notes |
|------|---------|-------|
| `ramp` | \|ω\| | Ram-Lak — sharpest, amplifies high-frequency noise |
| `shepp` | \|ω\| × sinc(ω) | Shepp-Logan — mild high-freq suppression |
| `parzen` | \|ω\| × B₄(2ω) | Parzen B-spline-4 — strongest smoothing, tapers to 0 at Nyquist |

**Chunking strategy** — same as before: the reconstruction is separable along `nz`.
`Chunking.gpu_batch` overlaps CPU↔GPU transfers with computation:

- `axis_inp=1` — chunk `psi_data` along its middle axis (vertical slices)
- `axis_out=0` — chunk `rec` along its first axis

In [ ]:
nchunk = 16   # vertical slices processed per GPU chunk

cl_tomo = Tomo(nobj, theta, mask_r=1.1)

# Pre-allocate the GPU ping-pong buffers:
#   input  double buffer: 2 × [ntheta, nchunk, nobj]  complex64
#   output double buffer: 2 × [nchunk,  nobj,  nobj]  complex64
nbytes = 2 * (ntheta * nchunk * nobj + nchunk * nobj**2) * np.dtype('complex64').itemsize
cl = Chunking(nbytes, nchunk)

rec = np.zeros([nobj, nobj, nobj], dtype='complex64')

@cl.gpu_batch(axis_out=0, axis_inp=1, nout=1)
def _fbp(cl, rec, psi_data):
    # psi_data: [ntheta, nchunk, nobj] on GPU (complex64)
    # rec:      [nchunk, nobj, nobj]   on GPU (complex64, in-place)
    rec[:] = cl_tomo.fbp(psi_data, filter_name)

_fbp(cl, rec, psi_data)

mshow_complex(rec[rec.shape[0] // 2], show)
mshow_complex(rec[rec.shape[0] // 2,
                  nobj // 2 - nobj // 8 : nobj // 2 + nobj // 8,
                  nobj // 2 - nobj // 8 : nobj // 2 + nobj // 8], show)

### Save initial reconstruction to HDF5

Store real and imaginary parts separately under keys that encode the Paganin parameter and bin
level, e.g. `obj_init_re20_2`.  Step 6 reads these as the starting guess for the iterative solver.

In [ ]:
with h5py.File(f'{path_out}/data.h5', 'a') as fid:
    for key in [f'/exchange/obj_init_re{paganin}_{bin}', f'/exchange/obj_init_imag{paganin}_{bin}']:
        if key in fid:
            del fid[key]
    fid.create_dataset(f'/exchange/obj_init_re{paganin}_{bin}', data=rec.real)
    fid.create_dataset(f'/exchange/obj_init_imag{paganin}_{bin}', data=rec.imag)